In [1]:
from utils import prepare_dataframe

df = prepare_dataframe("merged.csv",False)
resulting_label_count = df['labels'].nunique()
df

labels
enhancement    37586
bug            32864
clean          17263
performance     5392
refactor        4596
Name: count, dtype: int64
labels
enhancement    37586
bug            32864
clean          17263
Name: count, dtype: int64


,full_repo_name,labels,patch,index
1909,google/gson,enhancement,"@@ -289,24 +289,34 @@ public final T fromJsonT...",1909
1910,google/gson,enhancement,"@@ -28,23 +28,50 @@\n public class TypeAdapter...",1910
1947,google/conscrypt,enhancement,"@@ -1017,6 +1017,8 @@ public SSLEngineResult w...",1947
1948,google/conscrypt,enhancement,"@@ -293,13 +293,15 @@ public void removeValue(...",1948
1949,google/conscrypt,enhancement,"@@ -22,6 +22,7 @@\n import java.net.InetSocket...",1949
...,...,...,...,...
96453,apache/drill,clean,"@@ -53,7 +53,6 @@\n import org.junit.experimen...",96453
96454,apache/drill,clean,"@@ -25,14 +25,17 @@\n import org.apache.drill....",96454
96455,apache/drill,clean,"@@ -75,6 +75,8 @@ public static void initSplun...",96455
96456,apache/drill,clean,"@@ -17,6 +17,7 @@\n */\n package org.apache.d...",96456


In [2]:
embedding_directory = 'embeddings/GCB_Pooler_embeddings_patch_before_largewithcodedata_Xlimit3label'
model_directory = 'models/two_pipelines_StScaler_pooler_6layers_invertednn_100epoch_batch_normalization+17263.pth'

In [3]:
import pandas as pd
# Df@te saddece apache repolari olacak şekilde eleme yapildi
df = df[df['full_repo_name'].str.startswith('apache', na=False)]

# Count the occurrences of each unique value in the 'lables' column
label_counts = df['labels'].value_counts()
print(label_counts)

### Limit df size for testing purposes ###
label_enh = df[df['labels'] == 'enhancement']
label_bug = df[df['labels'] == 'bug']
label_cln = df[df['labels'] == 'clean']

min = len(label_cln)
sampled_enh = label_enh.sample(n=min, random_state=1)
sampled_bug = label_bug.sample(n=min, random_state=1)
sampled_cln = label_cln.sample(n=min, random_state=1)

df = pd.concat([sampled_enh, sampled_bug, sampled_cln])

# Count the occurrences of each unique value in the 'lables' column
label_counts = df['labels'].value_counts()
print(label_counts)

labels
bug            17934
enhancement    16113
clean          15336
Name: count, dtype: int64
labels
enhancement    15336
bug            15336
clean          15336
Name: count, dtype: int64


In [4]:
import torch

# One-hot encoding ile labels sütununu dönüştürme
df_encoded_labels = pd.get_dummies(df, columns=['labels'])
df_encoded_repo = pd.get_dummies(df, columns=['full_repo_name'])

# Özellikler ve etiketler
X_repo = df_encoded_repo.drop(columns=['labels','patch','index'])
X_repo = X_repo.astype(int)

# y = df_encoded.drop(columns=['content_before', 'content_after', 'tokens'])
y = df_encoded_labels.drop(columns=['patch', 'full_repo_name','index'])
y = y.astype(int) # convert it 0,1     instead of true, false 

df['label_tensor'] =[torch.tensor(row,dtype=torch.float32) for row in y.values] 
df['repo_name_tensor'] =[torch.tensor(row,dtype=torch.float32) for row in X_repo.values] 

df = df.drop(columns=['full_repo_name', 'labels','patch'])



df

,index,label_tensor,repo_name_tensor
56446,56446,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
57335,57335,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
79405,79405,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
45175,45175,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
42765,42765,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
...,...,...,...
63863,63863,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
71156,71156,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
78185,78185,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
42537,42537,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."


In [5]:
from utils import load_embeddings_from_directory_for_df
from sklearn.model_selection import train_test_split
import pandas as pd

df = load_embeddings_from_directory_for_df(directory_path=embedding_directory,df=df)


  0%|          | 0/46008 [00:00<?, ?it/s]

In [6]:
# Her sütundaki verinin tipini ve ilk birkaç satırdaki şekil bilgilerini inceleyelim
print("Embedding column types and shapes:")
print(df['embedding'].apply(lambda x: (type(x), x.shape if isinstance(x, torch.Tensor) else "Not a tensor")).head())

print("\nRepo name tensor column types and shapes:")
print(df['repo_name_tensor'].apply(lambda x: (type(x), x.shape if isinstance(x, torch.Tensor) else "Not a tensor")).head())

print("\nLabel tensor column types and shapes:")
print(df['label_tensor'].apply(lambda x: (type(x), x.shape if isinstance(x, torch.Tensor) else "Not a tensor")).head())


Embedding column types and shapes:
56446    (<class 'torch.Tensor'>, (768,))
57335    (<class 'torch.Tensor'>, (768,))
79405    (<class 'torch.Tensor'>, (768,))
45175    (<class 'torch.Tensor'>, (768,))
42765    (<class 'torch.Tensor'>, (768,))
Name: embedding, dtype: object

Repo name tensor column types and shapes:
56446    (<class 'torch.Tensor'>, (100,))
57335    (<class 'torch.Tensor'>, (100,))
79405    (<class 'torch.Tensor'>, (100,))
45175    (<class 'torch.Tensor'>, (100,))
42765    (<class 'torch.Tensor'>, (100,))
Name: repo_name_tensor, dtype: object

Label tensor column types and shapes:
56446    (<class 'torch.Tensor'>, (3,))
57335    (<class 'torch.Tensor'>, (3,))
79405    (<class 'torch.Tensor'>, (3,))
45175    (<class 'torch.Tensor'>, (3,))
42765    (<class 'torch.Tensor'>, (3,))
Name: label_tensor, dtype: object


In [22]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. embedding ve repo_name_tensor tensorlarını numpy formatına çeviriyoruz ve birleştiriyoruz
df['embedding'] = df['embedding'].apply(lambda x: x.numpy() if isinstance(x, torch.Tensor) else x)
df['repo_name_tensor'] = df['repo_name_tensor'].apply(lambda x: x.numpy() if isinstance(x, torch.Tensor) else x)

# Özellikleri birleştiriyoruz
X = np.vstack(df.apply(lambda row: np.concatenate([row['embedding'], row['repo_name_tensor']]), axis=1))

# 2. Label tensorundaki üç elemanı argmax ile tek bir sınıf etiketine çeviriyoruz
y = df['label_tensor'].apply(lambda x: x.numpy().argmax() if isinstance(x, torch.Tensor) else x).values

# 3. Veriyi eğitim ve test setlerine ayırma
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3074)

# 4. XGBoost modelini tanımlama
model = xgb.XGBClassifier(objective="multi:softmax", num_class=3, eval_metric="mlogloss")

# 5. Modeli eğitme
model.fit(X_train, y_train)

# 6. Test verisinde tahmin yapma
y_pred = model.predict(X_test)

# 7. Doğruluk oranını hesaplama
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy (Reponame + Embedding LHS)", accuracy)


Accuracy (Reponame + Embedding LHS) 0.80580308628559


In [18]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. embedding tensorlarını numpy formatına çeviriyoruz
df['embedding'] = df['embedding'].apply(lambda x: x.numpy() if isinstance(x, torch.Tensor) else x)

# Yalnızca embedding sütununu kullanarak özellik matrisi oluşturma
X = np.vstack(df['embedding'].values)

# Label tensorundaki üç elemanı argmax ile tek bir sınıf etiketine çeviriyoruz
y = df['label_tensor'].apply(lambda x: x.numpy().argmax() if isinstance(x, torch.Tensor) else x).values

# 2. Veriyi eğitim ve test setlerine ayırma
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. XGBoost modelini tanımlama
model = xgb.XGBClassifier(objective="multi:softmax", num_class=3, eval_metric="mlogloss")

# 4. Modeli eğitme
model.fit(X_train, y_train)

# 5. Test verisinde tahmin yapma
y_pred = model.predict(X_test)

# 6. Doğruluk oranını hesaplama
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy (Only Embedding Pooler):", accuracy)


Accuracy (Only Embedding Pooler): 0.730602043034123


In [17]:
X = np.vstack(df['embedding'].values)
np.min(X) 

-0.8153791

In [15]:
np.vstack(df['repo_name_tensor']).shape

(46008, 100)

In [1]:
from utils import prepare_dataframe
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from utils import load_embeddings_from_directory_for_df
import pandas as pd
import torch

#LAST HIDDEN STATE
embedding_directory = 'embeddings/GCB_LHS_embeddings_patch_before_largewithcodedata_Xlimit3label'


df = prepare_dataframe("merged.csv",False)
resulting_label_count = df['labels'].nunique()



# One-hot encoding ile labels sütununu dönüştürme
df_encoded_labels = pd.get_dummies(df, columns=['labels'])
df_encoded_repo = pd.get_dummies(df, columns=['full_repo_name'])

# Özellikler ve etiketler
X_repo = df_encoded_repo.drop(columns=['labels','patch','index'])
X_repo = X_repo.astype(int)

# y = df_encoded.drop(columns=['content_before', 'content_after', 'tokens'])
y = df_encoded_labels.drop(columns=['patch', 'full_repo_name','index'])
y = y.astype(int) # convert it 0,1     instead of true, false 

df['label_tensor'] =[torch.tensor(row,dtype=torch.float32) for row in y.values] 
df['repo_name_tensor'] =[torch.tensor(row,dtype=torch.float32) for row in X_repo.values] 

df = df.drop(columns=['full_repo_name', 'labels','patch'])

df = load_embeddings_from_directory_for_df(directory_path=embedding_directory,df=df)

# 1. embedding ve repo_name_tensor tensorlarını numpy formatına çeviriyoruz ve birleştiriyoruz
df['embedding'] = df['embedding'].apply(lambda x: x.numpy() if isinstance(x, torch.Tensor) else x)
df['repo_name_tensor'] = df['repo_name_tensor'].apply(lambda x: x.numpy() if isinstance(x, torch.Tensor) else x)

# Özellikleri birleştiriyoruz
X = np.vstack(df.apply(lambda row: np.concatenate([row['embedding'], row['repo_name_tensor']]), axis=1))

# 2. Label tensorundaki üç elemanı argmax ile tek bir sınıf etiketine çeviriyoruz
y = df['label_tensor'].apply(lambda x: x.numpy().argmax() if isinstance(x, torch.Tensor) else x).values

# 3. Veriyi eğitim ve test setlerine ayırma
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. XGBoost modelini tanımlama
model = xgb.XGBClassifier(objective="multi:softmax", num_class=3, eval_metric="mlogloss")

# 5. Modeli eğitme
model.fit(X_train, y_train)

# 6. Test verisinde tahmin yapma
y_pred = model.predict(X_test)

# 7. Doğruluk oranını hesaplama
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy (reponame + embedding LHS)", accuracy)



labels
enhancement    37586
bug            32864
clean          17263
performance     5392
refactor        4596
Name: count, dtype: int64
labels
enhancement    37586
bug            32864
clean          17263
Name: count, dtype: int64
labels
bug            17934
enhancement    16113
clean          15336
Name: count, dtype: int64
labels
enhancement    15336
bug            15336
clean          15336
Name: count, dtype: int64


  0%|          | 0/46008 [00:00<?, ?it/s]

: 

In [ ]:
import gc
import torch

df_train = df[df['index'].isin(X_index_train['index'].values)].reset_index(drop=True)
df_val = df[df['index'].isin(X_index_val['index'].values)].reset_index(drop=True)
df_test = df[df['index'].isin(X_index_test['index'].values)].reset_index(drop=True)

torch.cuda.empty_cache()

df_train

In [ ]:
from torch.utils.data import DataLoader, Dataset
from utils import load_embedding_from_directory

# Custom Dataset Class to handle the dataframe
class CodeDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.id_list = df['index'].values 

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
    
        index = self.id_list[idx]  

        tensor_3d = self.df.loc[self.df['index'] == index, 'embedding'].iloc[0]
        repo_name_tensor = self.df.loc[self.df['index'] == index, 'repo_name_tensor'].iloc[0]
        label_tensor = self.df.loc[self.df['index'] == index, 'label_tensor'].iloc[0]
        
        #print(f"{repo_name_tensor.shape} {tensor_3d.shape} {label_tensor.shape}")
        #print("-------------------------------")
        
        #Flattening
        #embedding = torch.flatten(tensor_3d)
        embedding = tensor_3d
        return embedding, label_tensor, repo_name_tensor, index


In [ ]:
# Create DataLoader

# Creating a DataLoader
def create_dataloader(df, batch_size=32, shuffle=True):
    dataset = CodeDataset(df)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

batch_size = 128
test_loader = create_dataloader(df=df_test, batch_size=batch_size)
train_loader = create_dataloader(df=df_train, batch_size=batch_size)
val_loader = create_dataloader(df=df_val,batch_size=batch_size)

full_loader_for_scaler = create_dataloader(df=df, batch_size=512)

In [ ]:
from sklearn.preprocessing import StandardScaler
import tqdm

scaler = StandardScaler()

for batch_X, batch_y, batch_X_repo, idx in tqdm.tqdm_notebook(full_loader_for_scaler):
    scaler.partial_fit(batch_X)

In [ ]:
import joblib

# Save scaler into file
joblib.dump(scaler, 'scalers/scaler_pooler_3labels_15300.pkl')

In [ ]:
import joblib

# Load scaler from file
scaler = joblib.load('scalers/scaler_pooler_3labels_15300.pkl')

In [ ]:
import torch.nn as nn

# Define the ANN model
class CodeDiffNN(nn.Module):
    def __init__(self, code_input_size, repo_input_size, hidden_sizes_embedding_pipeline,hidden_sizes_reponame_pipeline,hidden_sizes_merged_pipeline, output_size, dropout_rate):
        super(CodeDiffNN, self).__init__()
        
        # Pipeline for code embeddings
        self.code_fc1 = nn.Linear(code_input_size, hidden_sizes_embedding_pipeline[0])
        self.code_bn1 = nn.BatchNorm1d(hidden_sizes_embedding_pipeline[0])
        self.code_relu1 = nn.ReLU()
        self.code_dropout1 = nn.Dropout(dropout_rate)
        
        self.code_fc2 = nn.Linear(hidden_sizes_embedding_pipeline[0], hidden_sizes_embedding_pipeline[1])
        self.code_bn2 = nn.BatchNorm1d(hidden_sizes_embedding_pipeline[1])
        self.code_relu2 = nn.ReLU()
        self.code_dropout2 = nn.Dropout(dropout_rate)
        
        self.code_fc3 = nn.Linear(hidden_sizes_embedding_pipeline[1], hidden_sizes_embedding_pipeline[2])
        self.code_bn3 = nn.BatchNorm1d(hidden_sizes_embedding_pipeline[2])
        self.code_relu3 = nn.ReLU()
        self.code_dropout3 = nn.Dropout(dropout_rate)  
        
        
        # Pipeline for repo names
        self.repo_fc1 = nn.Linear(repo_input_size, hidden_sizes_reponame_pipeline[0])
        self.repo_bn1 = nn.BatchNorm1d(hidden_sizes_reponame_pipeline[0])
        self.repo_relu1 = nn.ReLU()
        self.repo_dropout1 = nn.Dropout(dropout_rate)
        
        self.repo_fc2 = nn.Linear(hidden_sizes_reponame_pipeline[0], hidden_sizes_reponame_pipeline[1])
        self.repo_bn2 = nn.BatchNorm1d(hidden_sizes_reponame_pipeline[1])
        self.repo_relu2 = nn.ReLU()
        self.repo_dropout2 = nn.Dropout(dropout_rate)

        
        # Combining the outputs of both pipelines
        combined_input_size = hidden_sizes_embedding_pipeline[2] + hidden_sizes_reponame_pipeline[1]   # Since we're combining two pipelines
        self.combined_fc1 = nn.Linear(combined_input_size, hidden_sizes_merged_pipeline[0])
        self.combined_bn1 = nn.BatchNorm1d(hidden_sizes_merged_pipeline[0])
        self.combined_relu1 = nn.ReLU()
        self.combined_dropout1 = nn.Dropout(dropout_rate)

        self.combined_fc2  = nn.Linear(hidden_sizes_merged_pipeline[0], hidden_sizes_merged_pipeline[1])
        self.combined_bn2 = nn.BatchNorm1d(hidden_sizes_merged_pipeline[1])
        self.combined_relu2  = nn.ReLU()
        self.combined_dropout2 = nn.Dropout(dropout_rate)
        
        self.combined_fc3  = nn.Linear(hidden_sizes_merged_pipeline[1], hidden_sizes_merged_pipeline[2])
        self.combined_bn3 = nn.BatchNorm1d(hidden_sizes_merged_pipeline[2])
        self.combined_relu3  = nn.ReLU()
        self.combined_dropout3 = nn.Dropout(dropout_rate)
        
        self.combined_fc4  = nn.Linear(hidden_sizes_merged_pipeline[2], hidden_sizes_merged_pipeline[3])
        self.combined_bn4 = nn.BatchNorm1d(hidden_sizes_merged_pipeline[3])
        self.combined_relu4  = nn.ReLU()
        self.combined_dropout4 = nn.Dropout(dropout_rate)

        # Output layer
        self.fc_out = nn.Linear(hidden_sizes_merged_pipeline[3], output_size)
        
    def forward(self, code_x, repo_x):
        # Process code embeddings
        code_x = self.code_fc1(code_x)
        code_x = self.code_bn1(code_x)
        code_x = self.code_relu1(code_x)
        code_x = self.code_dropout1(code_x)
        
        code_x = self.code_fc2(code_x)
        code_x = self.code_bn2(code_x)
        code_x = self.code_relu2(code_x)
        code_x = self.code_dropout2(code_x)
        
        code_x = self.code_fc3(code_x)
        code_x = self.code_bn3(code_x)
        code_x = self.code_relu3(code_x)
        code_x = self.code_dropout3(code_x)
        
        # Process repo names
        repo_x = self.repo_fc1(repo_x)
        repo_x = self.repo_bn1(repo_x)
        repo_x = self.repo_relu1(repo_x)
        repo_x = self.repo_dropout1(repo_x)
        
        repo_x = self.repo_fc2(repo_x)
        repo_x = self.repo_bn2(repo_x)
        repo_x = self.repo_relu2(repo_x)
        repo_x = self.repo_dropout2(repo_x)
        
        # Combine both outputs
        combined_x = torch.cat((code_x, repo_x), dim=1)
        
        # Process the combined output
        combined_x = self.combined_fc1(combined_x)
        combined_x = self.combined_bn1(combined_x)
        combined_x = self.combined_relu1(combined_x)
        combined_x = self.combined_dropout1(combined_x)

        combined_x = self.combined_fc2(combined_x)
        combined_x = self.combined_bn2(combined_x)
        combined_x = self.combined_relu2(combined_x)
        combined_x = self.combined_dropout2(combined_x)
        
        combined_x = self.combined_fc3(combined_x)
        combined_x = self.combined_bn3(combined_x)
        combined_x = self.combined_relu3(combined_x)
        combined_x = self.combined_dropout3(combined_x)
        
        combined_x = self.combined_fc4(combined_x)
        combined_x = self.combined_bn4(combined_x)
        combined_x = self.combined_relu4(combined_x)
        combined_x = self.combined_dropout4(combined_x)
        
        # Final output
        output = self.fc_out(combined_x)
        return output


In [ ]:
import torch.optim as optim
# Define model parameters

code_embedding_dim = df_train['embedding'][5].shape[0]

repo_input_size = len(df_train['repo_name_tensor'][0])
#hidden_sizes = [1024, 512, 256, 128]
hidden_sizes_embedding_pipeline = [1024, 2048, 4096]
hidden_sizes_reponame_pipeline = [512, 1024]
hidden_sizes_merged_pipeline = [2048, 1024, 512, 256]
output_size = resulting_label_count
dropout_rate = 0.05

device = "cuda:0"
model = CodeDiffNN(code_embedding_dim, repo_input_size, hidden_sizes_embedding_pipeline,hidden_sizes_reponame_pipeline,hidden_sizes_merged_pipeline, output_size, dropout_rate).to(device)
# model = SinglePipelineCodeDiffNN(code_embedding_dim, hidden_sizes, output_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)

# Decrease learning rate automatically on plateau
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=400, factor=0.1, verbose=True)

In [ ]:
print(resulting_label_count)
print(code_embedding_dim)
torch.cuda.empty_cache()

In [ ]:
# Function to calculate accuracy
def calculate_accuracy(outputs, labels):
    _, predicted = torch.max(outputs, 1)  # Get the index of the max log-probability
    # labels_1d = torch.argmax(labels, dim=1) # turn into 1D tensor
    correct = (predicted == labels).sum().item()
    accuracy = correct / labels.size(0)  # Accuracy = number of correct predictions / total number of samples
    return accuracy

In [ ]:
import tqdm

def train_with_data_loader(model,loader):
    
    running_loss = 0.0
    running_accuracy = 0.0
    
    for batch_X, batch_y, batch_X_repo, idx in tqdm.tqdm_notebook(loader):
        
        batch_X = scaler.transform(batch_X)
        batch_X = torch.tensor(batch_X,dtype=torch.float32)
        
        # Forward pass        
        batch_y = batch_y.to(device)
        batch_X = batch_X.to(device)
        batch_X_repo = batch_X_repo.to(device)
        
        batch_y = torch.argmax(batch_y, dim=1)

        # outputs = model(batch_X)
        outputs = model(batch_X, batch_X_repo)
        loss = criterion(outputs, batch_y)
    
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        accuracy = calculate_accuracy(outputs, batch_y)
        
        # Update running loss and accuracy
        running_loss += loss.item()
        running_accuracy += accuracy
        
        
    val_loss = running_loss / len(loader)
    val_accuracy = running_accuracy / len(loader)
    
    return val_loss, val_accuracy

In [ ]:
def evaluate_with_data_loader(model,loader):
    
    running_loss = 0.0
    running_accuracy = 0.0
    
    for batch_X, batch_y, batch_X_repo, idx in loader:
        
        batch_X = scaler.transform(batch_X)
        batch_X = torch.tensor(batch_X,dtype=torch.float32)
        
        # Forward pass
        batch_y = batch_y.to(device)
        batch_X = batch_X.to(device)
        batch_X_repo = batch_X_repo.to(device)
        
        batch_y = torch.argmax(batch_y, dim=1)

        outputs = model(batch_X, batch_X_repo)
        # outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Calculate accuracy
        accuracy = calculate_accuracy(outputs, batch_y)
        
        # Update running loss and accuracy
        running_loss += loss.item()
        running_accuracy += accuracy
        
        
    val_loss = running_loss / len(loader)
    val_accuracy = running_accuracy / len(loader)
    
    # Adjust learning rate based on validation loss
    scheduler.step(val_loss)
    
    return val_loss, val_accuracy

In [ ]:
def test_with_data_loader(model,loader):
    
    running_loss = 0.0
    running_accuracy = 0.0
    predicted_y = []
    y = []
    
    for batch_X, batch_y, batch_X_repo, idx in loader:
        
        batch_X = scaler.transform(batch_X)
        batch_X = torch.tensor(batch_X,dtype=torch.float32)

        # Forward pass
        batch_y = batch_y.to(device)
        batch_X = batch_X.to(device)
        batch_X_repo = batch_X_repo.to(device)
        
        # index encoding
        batch_y_1D = torch.argmax(batch_y, dim=1)

        outputs = model(batch_X, batch_X_repo)
        # outputs = model(batch_X)
        loss = criterion(outputs, batch_y_1D)
        
        # Calculate accuracy
        accuracy = calculate_accuracy(outputs, batch_y_1D)
        
        # Update running loss and accuracy
        running_loss += loss.item()
        running_accuracy += accuracy

        predicted_y.append(outputs)
        y.append(batch_y)
        
    test_loss = running_loss / len(loader)
    test_accuracy = running_accuracy / len(loader)
    
    return test_loss, test_accuracy, torch.cat(predicted_y, dim=0), torch.cat(y, dim=0)

In [ ]:
import copy
import tqdm

# Train the model

# Initialize variables to track the best model and best accuracy
best_accuracy = 0.0

num_epochs = 100

for epoch in tqdm.tqdm_notebook(range(num_epochs)):
    model.train()  # Set the model to training mode
    
    # Print average loss and accuracy for the epoch
    avg_loss,avg_accuracy = train_with_data_loader(model= model,loader= train_loader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {avg_accuracy:.4f}')

    # Evaluate on validation data
    model.eval()  # Set the model to evaluation mode
    
    with torch.no_grad():  # Disable gradient computation for validation
        val_loss,val_accuracy = evaluate_with_data_loader(model=model,loader=val_loader)
        print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')
    
    # Check if the current model has the best validation accuracy
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        torch.save(model.state_dict(), model_directory)
    



In [ ]:
# Load model from file
model = CodeDiffNN(code_embedding_dim, repo_input_size, hidden_sizes_embedding_pipeline,hidden_sizes_reponame_pipeline, hidden_sizes_merged_pipeline, output_size, dropout_rate).to(device)
model.load_state_dict(torch.load(model_directory))

In [ ]:
with torch.no_grad():
    # Load best model from previous training cycle

    model.eval()
    test_loss,test_accuracy,test_outputs,y_test = test_with_data_loader(model=model,loader=test_loader)
    
    print(f'Accuracy on test data: {test_accuracy:.4f}')
    print(test_outputs.shape)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

_, predicted = torch.max(test_outputs.detach().cpu(), 1)  # Get the index of the max log-probability
labels_1d = torch.argmax(y_test, dim=1).cpu() # turn into 1D tensor

# Convert to numpy arrays
y_test_np = labels_1d.numpy()
predicted_np = predicted.numpy()

# Compute confusion matrix
cm = confusion_matrix(y_test_np, predicted_np)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title('Confusion Matrix for Multi-Class Classification')
plt.show()

print(predicted_np)
print(y_test_np)

In [ ]:
import gc
# Free up GPU memory (testing)
print(f"Cached GPU memory before: {torch.cuda.memory_reserved()} bytes")
del model
del test_outputs
del y_test
del predicted
del labels_1d
del val_loader, test_loader, train_loader
del cm
gc.collect()
torch.cuda.empty_cache()
print(f"Cached GPU memory after: {torch.cuda.memory_reserved()} bytes")